# Grid world benchmark

The stepping stone between the clip world (`01_gain_attribution.ipynb`) and Craftax. Same question: does a fix keep the
content that is predictable from the action but not yet learned, while the encoder still drops noise? Here the content
is discrete, relational and Craftax-shaped. Code: `gaf/env_grid.py`; the training code is shared with the clip world.

| group | grid world | R² without the action |
|---|---|---|
| **slow** | 16 inventory slots in the HUD (0–9, tile brightness, ±1 with p = 0.03 per frame) + the agent's position | ≈ 0.99 |
| **fast** | the agent's facing: every move action turns it, even when the move is blocked (yellow half of the agent tile) | ≈ 0.03 |
| **hard** | the hand: the material of the cell the agent faces after its action (empty / wood / stone / water), shown in the HUD slot after the inventory | ≈ 0.2 |
| **noise** | a green mob that jumps to a random free cell every frame, and 4 magenta tiles with a random brightness every frame | 0 |

Wood (brown), stone (grey) and water (blue) block movement; the map is new in every clip. Actions: noop, up, down, left,
right (one-hot, random policy), seen by the predictor only.

**Stages**
0. Criteria, written before any result.
1. Data: frames and sampling speed.
2. Premise, capacity (which D), and availability (what a SIGReg-only latent holds: `detached`).
3. Learnability on ground truth.
4. Baselines `lewm`, `sg`, `ema`: does the failure happen here?
5. The fix: B (`retain_open`), behind a short probe. (A and the head-start control are dropped: stage 2c.)
7. Noise against content: per part and per direction.
8. Stress (optional). 9. Handoff to Craftax.

**Seeds.** Every stage runs seed 0 only. The cell at the end (optional) repeats stages 4 and 5 for seeds 1 and 2 and
gives the seed means. Every run is resumable: finished runs in `OUT` are skipped, so after an interruption re-run the cell.

In [ ]:
%pip install -q -e ..
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
import time
from IPython.display import display
import pandas as pd
from gaf.env import GROUPS
from gaf.env_grid import GridCfg
from gaf.train import Cfg, capacity, premise, run_gt
from gaf import analysis as A

OUT = "/workspace/results/grid_benchmark"   # separate from the clip world's results
SEED = 0                          # every stage: one seed. The last cell adds EXTRA_SEEDS (optional)
EXTRA_SEEDS = (1, 2)
WORKERS = 3   # runs at once on the GPU (separate processes; progress in <run_id>/log.txt). 1 = sequential

# ---- the knobs ---------------------------------------------------------------------------
GRID = GridCfg(
    W=10, H=8, tile=4, T=3,              # 40x40 RGB: 10x8 map + 2 HUD rows; 2 context frames + 1 target
    n_slow=16, levels=10, hud_p=0.03,    # inventory: 16 slots, 0..9 as brightness, +-1 w.p. 0.03 per frame
    density=0.35, random_map=True,       # wood / stone / water on 35% of the cells, a new map per clip
    n_twinkle=4,                         # noise tiles, random brightness every frame
)
D = 32   # from the capacity sweep (stage 2b)
# the clip world's hyperparameters, unchanged (criterion 3: transfer without retuning)
COMMON = dict(env=GRID, D=D, batch=256, lr=1e-3, lam=0.1, wd=0.01, basis="excess",
              floor_steps=1500, floor_warm=True)
EARLY = (0, 25, 50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000, 4000)
BASE_STEPS, FIX_STEPS = 10_000, 20_000   # baselines / the full run of B
pd.set_option("display.precision", 3)
COLS = [f"r2_{g}" for g in GROUPS] + [f"mlp_r2_{g}" for g in GROUPS]


def cfg(arm, seed=SEED, **kw):
    return Cfg(arm=arm, seed=seed, **{**COMMON, **kw})


def after(k):   # early evals around a head-start switch at step k
    return tuple(sorted(set(EARLY) | {k + d for d in (0, 50, 100, 200, 500, 1000)}))


def rows(df, sels, seeds=(SEED,)):
    """Final R^2 per labelled run selector ({label: cfg fields}), mean over `seeds`."""
    out = {}
    for lab, sel in sels.items():
        d = A.select(df, **sel)
        d = d[d.seed.isin(seeds)]
        if len(d):
            out[lab] = d[COLS].mean()
    return pd.DataFrame(out).T

## 0 Criteria (fixed before running)
1. **The baseline fails:** in `sg` and `lewm`, fast and hard R² < 0.3, and the latent is made of slow-type content. With the random
   map that is the map as much as the HUD: slow R² (inventory + position) alone can be ≈ 0 while the latent is still a function of
   the map. Stage 4b measures it (`latent_determined_by`): `facing+hand` < 0.2 and `map` or `inventory+position` ≥ 0.5 (first guesses).
   If not, stop after stage 4 and add pressure (smaller D, lower `density`) before spending the 20k-step runs.
2. **A fix works:** fast and hard ≥ 0.8, slow ≥ `sg` − 0.1, noise ≤ 0.3. Per part (stage 7): the mob ≤ 0.3, i.e. sprite-like
   noise is dropped while the facing (also a sprite feature) is kept.
3. **Transfer:** criterion 2 with the clip world's hyperparameters unchanged. Retuning, if needed, is the result.
4. **Settled:** `A.drift` says `settled`; otherwise the numbers are provisional.

R² values are ridge readouts from z_t (seed 0); the MLP probe (`mlp_r2_*`) is alongside.

## 1 Data
Rows are clips, columns frames. Map: wood (brown), stone (grey), water (blue), floor (dark). White tile: the agent, its
yellow half is the facing. Green: the noise mob. Magenta: noise tiles. Bottom two rows: the inventory (16 slots), then
the hand slot (colour of the material in front of the agent).

In [ ]:
A.show_clips(GRID, n=4);

In [ ]:
from gaf.env import ClipEnv, EnvCfg
from gaf.env_grid import GridEnv
dev = "cuda" if torch.cuda.is_available() else "cpu"
for name, env in (("clip world", ClipEnv(EnvCfg(), dev)), ("grid world", GridEnv(GRID, dev))):
    env.sample(256)
    if dev == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    for _ in range(100):
        env.sample(256)
    if dev == "cuda":
        torch.cuda.synchronize()
    print(f"{name}: {(time.time() - t) * 10:.1f} ms per batch of 256 clips")

## 2 Premise, capacity, availability
**2a Premise.** Can slow content alone satisfy SIGReg? SIGReg of hand-made latents of size D (as in the clip world):
`slow+copies` close to `gaussian` = yes. Slow here is the inventory and the agent's position; the random map is further
slow content (constant within a clip) that this check leaves out, so it errs on the strict side.

In [ ]:
display(premise(GRID, D=D).to_frame().T)

**2b Capacity: which D?** The real encoder on rendered frames, *supervised* with the ground truth plus SIGReg, at several D.
A linear head reads slow, fast and hard from z (readout); the predictor with a linear head predicts the target frame's
factors (prediction). This is an upper bound for any training of that size: the smallest D where every group reaches its
plateau is the room the content needs. Targets are standardized and the errors summed over columns, so the supervision outweighs SIGReg (λ = 0.1) and SIGReg only shapes the latent. The real runs also spend dimensions on the map and on noise, so leave a margin.

**Result (first run on the pod, before the loss fix): D = 32.** Slow was complete from D = 24 (0.70 at D = 16: 18 slow
dims do not fit). Facing (≈ 0.02) and hand (≈ 0.5) did not change from D = 32 to 64: with the old loss (a mean over
columns) SIGReg pushed the one-hot content out at every D. Without SIGReg both are read at 0.99 within 600 steps, so they
are visible. The loss is fixed; re-running this cell is optional.

**What it means for the real runs.** Facing and hand are categorical, and SIGReg finds 4-point clusters expensive (2a:
677 for fast + hard). Without supervision, the SIGReg-only latent may hold little of them: stage 2c tells.

**Choosing D (if re-run).** The smallest D at which every `read_*` and `pred_*` is within 0.05 of its value at D = 64. If it
is above 32, set `D` in the config cell and re-run 2a. In the clip world, D = 32 matched the content, and at D = 64 the
retention of variant B never released.


In [ ]:
cap = capacity(GRID, Ds=(16, 24, 32, 48, 64), steps=5000)
last = cap.groupby("D").tail(1).set_index("D")
display(last[[c for c in last if c.startswith(("read_", "pred_"))] + ["sigreg"]])
ax = cap.pivot(index="step", columns="D", values="pred_hard").plot(figsize=(6, 3.4))
ax.set(ylabel="prediction R², hard", ylim=(0, 1.02));

**2c Availability (`detached`).** The encoder sees only SIGReg; the predictor trains on its detached latents. Over training:
what the latent holds (`r2_*`) and what the predictor makes of it (`pred_r2_*`). The step at which `pred_r2_hard` levels
off is hard's learn time in latent space. If hard or fast never enter this latent, no gate or head start can protect them
(the clip world's red-sprite problem), and only the reconstruction shadow (B) can bring them in.

**Result (pod, seed 0): the latent holds no facing (r2 ≈ 0 at every step) and almost no hand (≤ 0.02).** The encoder can
represent both (2b, supervised: 0.99), so SIGReg alone never selects them: they are categorical and tiny in pixels. Variant A
and the `sg` + head start control are therefore dropped; B's reconstruction shadow is the only route.

In [ ]:
R2C = [cfg("detached", steps=BASE_STEPS, early_eval=EARLY, tag="avail")]
A.execute(R2C, OUT, workers=WORKERS)
df = A.collect(OUT)
av = {"detached": dict(arm="detached", tag="avail")}
display(A.snapshot(df, OUT, av, steps=(100, 500, 1000, 2000, 3000, 5000, 10000),
                   cols=("r2_fast", "r2_hard", "pred_r2_fast", "pred_r2_hard")))
A.trace(df, OUT, av, cols=("r2_fast", "r2_hard", "pred_r2_fast", "pred_r2_hard"));

## 3 Learnability on ground truth
A predictor from the true context factors and actions to the target factors. Its inputs also hold the materials around the
agent (cells 1 and 2 away in each direction), which the hand depends on and a good latent would hold. Slow at once, fast
quickly, hard later, noise never. Without actions, fast and hard must collapse (≈ 0.03 and ≈ 0.2).

**The probe must be ahead.** As in the clip world: the warm floor probe adds `floor_steps` per refresh, so it is ahead of
the predictor when `floor_steps > floor_every`. The cell prints when the probe has trained as long as hard's learn time.

In [ ]:
gt = run_gt(GRID, steps=5000, desc="ground truth, with actions")
gt_noact = run_gt(GRID, steps=3000, actions=False, desc="ground truth, without actions")
ax = gt.plot(figsize=(6, 3.4), color=[A.GROUP_COLOR[g] for g in GROUPS])
ax.set(xlabel="step", ylabel="held-out R²", ylim=(-0.05, 1.02)); ax.legend(loc="lower right")
display(pd.concat({"with actions": gt.iloc[[0, 1, 3, 9, 19, -1]], "without actions": gt_noact.iloc[[-1]]}))
t90 = {g: (int(gt.index[(gt[g] >= 0.9 * gt[g].iloc[-1]).values.argmax()]) if gt[g].iloc[-1] > 0.5 else None)
       for g in GROUPS[:3]}
print("steps to 90% of final R² on ground truth:", t90)
c0 = Cfg(**COMMON)
if t90["hard"] is None:
    print("WARNING: hard is not learned even from ground truth: lower `density` or check the env.")
else:
    k = -(-t90["hard"] // c0.floor_steps) - 1
    print(f"warm probe has trained {t90['hard']} steps after refresh {k + 1} "
          f"(step {c0.floor_start + k * c0.floor_every}); the predictor needs ~{t90['hard']} steps on ground truth "
          "(more in latent space: stage 2c)")

## 4 Baselines
`lewm` (full gradient through both paths), `sg` (stop-grad target), `ema` (EMA target encoder), 10k steps with early evals.
Criterion 1: slow-type content kept, fast and hard dropped (see 4b: with the random map, slow R² alone is the wrong measure). **If neither `sg` nor `lewm` fails, stop here** (see criterion 1).

**First result (seed 0):** all three baselines lose the labelled slow content too (slow ≈ 0, `ema` 0.19) as well as facing, hand and
noise. The latent is a perfect, full-rank Gaussian (SIGReg 1.27, `rank_z` 31.4) and almost constant within a clip (`wb_ratio` ≈ 0). That
is the premise in its strongest form, not a broken run: the random map is slow content with more entropy than the HUD, and nothing
labels it, so the group R² cannot see where the latent went. 4b measures it.


In [ ]:
R4 = [cfg(a, steps=BASE_STEPS, early_eval=EARLY) for a in A.BASELINES]
A.execute(R4, OUT, workers=WORKERS)
df = A.collect(OUT)
t_base = A.table(df[df.arm.isin(A.BASELINES) & (df.tag == "") & (df.seed == SEED)])
display(t_base)
display(A.baseline_verdict(t_base))
A.bars(t_base); A.curves(df[df.seed == SEED], OUT, A.BASELINES);
display(A.snapshot(df, OUT, {a: dict(arm=a, tag="", seed=SEED) for a in A.BASELINES},
                   steps=(0, 50, 100, 200, 500, 1000, 3000, 10000), cols=("r2_fast", "r2_hard")))

**4b What is the latent made of?** A small MLP predicts z_t from a set of ground-truth factors (held-out R², fresh clips): 1 = the
latent is a function of that content, 0 = it carries nothing of it. Sets: the whole map, inventory + position, facing + hand, mob + twinkle,
and all together. The expected picture for the failure: `map` (and/or `inventory+position`) high, `facing+hand` and `mob+twinkle` ≈ 0.
Seconds per run; no training of the encoder.

In [ ]:
held = {a: cfg(a, steps=BASE_STEPS, early_eval=EARLY) for a in A.BASELINES}
held["detached"] = cfg("detached", steps=BASE_STEPS, early_eval=EARLY, tag="avail")
display(A.latent_determined_by(held, OUT))
sel = (df.seed == SEED) & df.arm.isin([*A.BASELINES, "detached"]) & df.tag.isin(["", "avail"])
display(df[sel][["arm", "sigreg", "rank_z", "wb_ratio", "pred_mse", "loss"]].round(3))

## 5 The fix: B (`retain_open`)
Stage 2c showed that a SIGReg-only latent holds no facing and almost no hand, so a head start (variant A) and the control
`sg` + head start cannot protect what is not there: both are dropped. Only B's reconstruction shadow can bring the content in.

**5a Probe, 5k steps.** `retain_open` with early evals. Read `sh_r2_fast` and `sh_r2_hard` (what the shadow holds) and
`r2_*` (the live encoder). If the shadow holds neither (< 0.3 by step 5000), the reconstruction does not reward them: raise their
salience in the env (a bigger or brighter facing marker, like the clip world's `a_fast`) and repeat from 2c.

**5b Full run, 20k steps,** only if the probe passes (`FORCE_FULL = True` runs it anyway). About 18 min alone (1.7× a plain run).

In [ ]:
R5A = [cfg("retain_open", steps=5000, early_eval=EARLY, tag="ret-probe")]
A.execute(R5A, OUT, workers=WORKERS)
df = A.collect(OUT)
PR = {"B probe": dict(arm="retain_open", tag="ret-probe", seed=SEED)}
display(A.snapshot(df, OUT, PR, steps=(500, 1000, 2000, 3000, 5000),
                   cols=("sh_r2_fast", "sh_r2_hard", "r2_fast", "r2_hard")))
A.trace(df, OUT, PR, cols=("sh_r2_fast", "sh_r2_hard", "r2_fast", "r2_hard"), xlog=False);

In [ ]:
FORCE_FULL = False
last = A.select(df, **PR["B probe"]).iloc[0]
ok = max(last.sh_r2_fast, last.sh_r2_hard) >= 0.3
print(f"probe, step 5000: shadow holds fast {last.sh_r2_fast:.2f}, hard {last.sh_r2_hard:.2f}; "
      f"live encoder fast {last.r2_fast:.2f}, hard {last.r2_hard:.2f}")
if ok or FORCE_FULL:
    A.execute([cfg("retain_open", steps=FIX_STEPS, early_eval=EARLY, tag="ret")], OUT, workers=WORKERS)
    df = A.collect(OUT)
else:
    print("the shadow holds neither facing nor hand: no full run. Raise the salience in the env (see 5a).")

**5a2 Frame-difference reconstruction.** The probes of 5a with more room (D = 64) or a lower map entropy (density 0.10) did not bring facing
or hand into the shadow (facing ≈ 0 in all four, hand ≤ 0.09): with plain pixel MSE the map carries 77% of the loss, facing 3.5%, hand 1%.
`rec_diff=True`: the shadow reconstructs x_t − x_{t−1} from the pair (z_{t−1}, z_t). The static map cancels, and facing and hand carry 15% and
5% of the loss. Two probes, 5k steps, the random map and D = 32 as in 5a: `shadow_rec=4` (compensates the ≈ 3.6× smaller loss) and
`shadow_rec=1` (same weight as 5a). `shadow_rec_mse` is the reconstruction term itself (of the differences here).
Read `sh_r2_fast` / `sh_r2_hard`: ≥ 0.3 and rising means the shadow holds them; then the full run (5b) is worth it. `sh_r2_slow` falls by
design (the shadow no longer reconstructs the static content).

In [ ]:
R5D = [cfg("retain_open", steps=5000, early_eval=EARLY, rec_diff=True, shadow_rec=4.0, tag="probe-diff"),
       cfg("retain_open", steps=5000, early_eval=EARLY, rec_diff=True, shadow_rec=1.0, tag="probe-diff-w1")]
A.execute(R5D, OUT, workers=WORKERS)
df = A.collect(OUT)
SD = {"plain frames (5a)": dict(arm="retain_open", tag="ret-probe"),
      "differences, shadow_rec 4": dict(arm="retain_open", tag="probe-diff"),
      "differences, shadow_rec 1": dict(arm="retain_open", tag="probe-diff-w1")}
for col in ("sh_r2_fast", "sh_r2_hard", "sh_r2_slow", "sh_r2_noise", "r2_fast", "r2_hard", "shadow_rec_mse"):
    print(col)
    display(A.snapshot(df, OUT, SD, steps=(500, 1000, 2000, 3000, 5000), cols=(col,)).rename(columns=lambda c: c.split("@")[1]))
A.trace(df, OUT, SD, cols=("sh_r2_fast", "sh_r2_hard", "r2_fast", "r2_hard"), xlog=False);

**5a2 result (seed 0, 5k steps).** The difference loss does not bring facing in (shadow ≈ 0 for both weights). The hand rises a little
(0.05 at `shadow_rec` 4, 0.015 at 1), noise rises (0.36), slow falls (0.36 → 0.16): the shadow now spends its room on change, but mostly
on the mob and twinkles. Six probes in total (D 32/64, density 0.35/0.10, frames/differences) have left facing ≈ 0.

**5a3 Is the information there, just not linearly readable?** All shadow readouts so far are ridge (linear) probes, and SIGReg makes the
latent rotation-invariant. Facing is a position-dependent feature (the marker sits wherever the agent is), so a linear readout can miss
it even when the latent holds it. The finished runs did not keep their shadow. These two reruns (same configs, new tags) also run an
MLP probe on the shadow (`sh_mlp_r2_*`) and save the shadow weights. Compare `sh_mlp_r2_*` (nonlinear) with `sh_r2_*` (linear).

In [ ]:
R5M = [cfg("retain_open", steps=5000, early_eval=EARLY, tag="probe-mlp"),
       cfg("retain_open", steps=5000, early_eval=EARLY, rec_diff=True, shadow_rec=4.0, tag="probe-diff-mlp")]
A.execute(R5M, OUT, workers=WORKERS)
df = A.collect(OUT)
m = df[df.tag.isin(["probe-mlp", "probe-diff-mlp"])].set_index("tag")
display(m[[f"sh_r2_{g}" for g in GROUPS] + [f"sh_mlp_r2_{g}" for g in GROUPS] + ["shadow_rec_mse"]].round(3))

In [ ]:
SEL = {"sg": dict(arm="sg", tag=""), "lewm": dict(arm="lewm", tag=""), "ema": dict(arm="ema", tag=""),
       "B: retain_open": dict(arm="retain_open", tag="ret")}
t5 = rows(df, SEL)
display(t5)


def verdict(t, ref="sg", keep=0.8, drop=0.3, slack=0.1):
    r = t.loc[ref]
    return pd.DataFrame({lab: {f"fast >= {keep}": t.loc[lab, "r2_fast"] >= keep,
                               f"hard >= {keep}": t.loc[lab, "r2_hard"] >= keep,
                               f"noise <= {drop}": t.loc[lab, "r2_noise"] <= drop,
                               f"slow >= {ref} - {slack}": t.loc[lab, "r2_slow"] >= r.r2_slow - slack}
                         for lab in t.index if lab not in ("sg", "lewm", "ema")}).T


HAVE_B = len(A.select(df, arm="retain_open", tag="ret", seed=SEED)) > 0
if HAVE_B:
    display(verdict(t5))
    print("settled? change and swing over the last 4000 steps")
    display(A.drift(df, OUT, {"B: retain_open": dict(arm="retain_open", tag="ret", seed=SEED)}))
    A.trace(df, OUT, {"sg": dict(arm="sg", tag="", seed=SEED),
                      "B: retain_open": dict(arm="retain_open", tag="ret", seed=SEED)},
            cols=("r2_fast", "r2_hard", "r2_noise", "r2_slow"));

**Reading stages 4–5**
- `sg`, `lewm` and `ema` hold no facing or hand, like `detached` in 2c: the failure is availability, not pruning. If one of
  them does keep facing (prediction pressure brings it in), note which.
- Probe: the shadow holds facing and hand → run the full B and read stage 7. It holds neither → reconstruction does not reward
  them (8 pixels of the frame): raise their salience in the env, then repeat from 2c. Craftax's categorical detail is just as small.
- Full B keeps fast and hard and drops noise: the clip-world result transfers. Check `settled` before calling it.
- B keeps noise: the release rule does not free the noise directions (`ref_unsettled` high, stage 7).
- Slow falls while fast and hard are kept: the retention costs live capacity (the random map is further content).

## 7 Noise against content
Per part, from the final z_t (ridge R²): inside slow the inventory and the position, inside noise the mob (a sprite, like
the facing) and the tiles. The key test is the mob: it looks like content and only its dynamics say it is noise. Then the
gate and the reference per direction: which directions stay protected, and does ρ separate content from noise without labels?

In [ ]:
from pathlib import Path
P7 = {"detached": cfg("detached", steps=BASE_STEPS, early_eval=EARLY, tag="avail"),
      "sg": cfg("sg", steps=BASE_STEPS, early_eval=EARLY),
      "B probe (5k)": cfg("retain_open", steps=5000, early_eval=EARLY, tag="ret-probe"),
      "B: retain_open": cfg("retain_open", steps=FIX_STEPS, early_eval=EARLY, tag="ret")}
display(A.parts_r2({k: c for k, c in P7.items() if (Path(OUT) / c.run_id / "summary.json").exists()}, OUT))

In [ ]:
ret = A.select(df, arm="retain_open", tag="ret", seed=SEED)
if not len(ret):
    ret = A.select(df, arm="retain_open", tag="ret-probe", seed=SEED)
    print("no full run yet: showing the 5k probe")
ref_cols = (["ref_auc"] + [f"ref_rho_{g}" for g in GROUPS] + [f"ref_n_{g}" for g in GROUPS]
            + ["ref_w", "ref_unsettled", "ref_free"] + [f"sh_r2_{g}" for g in GROUPS])
display(ret[ref_cols].round(3))
A.gate_map(OUT, ret.run_id.iloc[0]);

**Reading stage 7**
- Mob low and facing high in B: the method tells sprite-like noise from sprite-like content by dynamics alone.
- Mob kept: the open path never frees its directions. Look at `ref_rho_noise` and `ref_unsettled`: a mob direction that
  stays "unsettled" is retained by the shadow's rule, not by the live predictor.
- `ref_auc` < 0.9: ρ does not separate content from noise here; read the gate maps before trusting the B result.
- Inventory falls in B while position stays: the retention costs live capacity; compare with stage 2b.

## 8 Stress (optional)
One seed each, 20k steps. More room (D = 64, where the clip world's retention never released), a fixed map (the pattern
moves from the frame into the dynamics), and a denser map (the hand changes more often and is harder to learn).

In [ ]:
RUN_STRESS = False
if RUN_STRESS:
    R8 = [cfg("retain_open", steps=FIX_STEPS, D=64, tag="ret-D64"),
          cfg("retain_open", steps=FIX_STEPS, env=GridCfg(**{**GRID.__dict__, "random_map": False}), tag="ret-fixed"),
          cfg("retain_open", steps=FIX_STEPS, env=GridCfg(**{**GRID.__dict__, "density": 0.5}), tag="ret-dense")]
    A.execute(R8, OUT, workers=WORKERS)
    df = A.collect(OUT)
    display(rows(df, {t: dict(arm="retain_open", tag=t) for t in ("ret", "ret-D64", "ret-fixed", "ret-dense")}))
    display(A.drift(df, OUT, {t: dict(arm="retain_open", tag=t, seed=SEED) for t in ("ret-D64", "ret-fixed", "ret-dense")}))

## 9 Handoff to Craftax
| grid world | Craftax | note |
|---|---|---|
| inventory slots (slow) | inventory and status bars | at reset all zero: batch variance only from trajectory diversity, so sample from mid-episode states |
| agent position (slow) | the map scrolls around the player (egocentric) | one-tile moves are mostly predictable without the action |
| facing (fast) | player sprite direction | changes with nearly every move action |
| hand (hard) | the block in front of the player, what `do` acts on | depends on the action and the blocks around the player |
| mob, twinkle tiles (noise) | mob movement, animated tiles, daylight | partly predictable in Craftax: expect a nonzero floor |

Carry over: the config of the variant that met the criteria, `D` from stage 2b (scaled with the content), the logs
`ref_auc`, `ref_unsettled`, `n_S`, and the settledness check. Labels for probes come from Craftax's symbolic state.

## All seeds (optional)
Repeats stages 4 and 5b (full B, no probe) for `EXTRA_SEEDS` and gives the seed means (stage 2c for the availability of every seed too). With
`WORKERS = 3`, about 3× the time of stages 4–5. The verdicts below use the means over all three seeds.

In [ ]:
RUN_ALL_SEEDS = False
if RUN_ALL_SEEDS:
    extra = [c for s in EXTRA_SEEDS for c in (
        [cfg("detached", s, steps=BASE_STEPS, early_eval=EARLY, tag="avail")]
        + [cfg(a, s, steps=BASE_STEPS, early_eval=EARLY) for a in A.BASELINES]
        + [cfg("retain_open", s, steps=FIX_STEPS, early_eval=EARLY, tag="ret")])]
    A.execute(extra, OUT, workers=WORKERS)
    df = A.collect(OUT)
    seeds = (SEED, *EXTRA_SEEDS)
    t_all = rows(df, SEL, seeds=seeds)
    sd = pd.DataFrame({lab: A.select(df, **s)[lambda d: d.seed.isin(seeds)][COLS].std()
                       for lab, s in SEL.items()}).T
    print("seed mean"); display(t_all)
    print("seed sd"); display(sd.round(3))
    display(A.baseline_verdict(A.table(df[df.arm.isin(A.BASELINES) & (df.tag == "")])))
    display(verdict(t_all))
    print("per seed")
    display(df[df.tag.isin(["", "ret"])][["arm", "tag", "seed"] + COLS]
            .sort_values(["arm", "tag", "seed"]).round(3))

## All runs

In [ ]:
df = A.collect(OUT)
df.to_csv(f"{OUT}/summary.csv", index=False)
print(len(df), "runs;", f"{df.seconds.sum() / 3600:.1f} GPU-hours")